# EBAC - Regressão II - regressão múltipla

## Tarefa I

#### Previsão de renda II

Vamos continuar trabalhando com a base 'previsao_de_renda.csv', que é a base do seu próximo projeto. Vamos usar os recursos que vimos até aqui nesta base.

|variavel|descrição|
|-|-|
|data_ref                | Data de referência de coleta das variáveis |
|index                   | Código de identificação do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui veículo|
|posse_de_imovel         | Indica se o cliente possui imóvel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instrução do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de residência do cliente (própria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na residência|
|renda                   | Renda em reais|

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('previsao_de_renda.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             15000 non-null  int64  
 1   data_ref               15000 non-null  object 
 2   id_cliente             15000 non-null  int64  
 3   sexo                   15000 non-null  object 
 4   posse_de_veiculo       15000 non-null  bool   
 5   posse_de_imovel        15000 non-null  bool   
 6   qtd_filhos             15000 non-null  int64  
 7   tipo_renda             15000 non-null  object 
 8   educacao               15000 non-null  object 
 9   estado_civil           15000 non-null  object 
 10  tipo_residencia        15000 non-null  object 
 11  idade                  15000 non-null  int64  
 12  tempo_emprego          12427 non-null  float64
 13  qt_pessoas_residencia  15000 non-null  float64
 14  renda                  15000 non-null  float64
dtypes:

1. Separe a base em treinamento e teste (25% para teste, 75% para treinamento).
2. Rode uma regularização *ridge* com alpha = [0, 0.001, 0.005, 0.01, 0.05, 0.1] e avalie o $R^2$ na base de testes. Qual o melhor modelo?
3. Faça o mesmo que no passo 2, com uma regressão *LASSO*. Qual método chega a um melhor resultado?
4. Rode um modelo *stepwise*. Avalie o $R^2$ na vase de testes. Qual o melhor resultado?
5. Compare os parâmetros e avalie eventuais diferenças. Qual modelo você acha o melhor de todos?
6. Partindo dos modelos que você ajustou, tente melhorar o $R^2$ na base de testes. Use a criatividade, veja se consegue inserir alguma transformação ou combinação de variáveis.
7. Ajuste uma árvore de regressão e veja se consegue um $R^2$ melhor com ela.

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore') # Para ignorar avisos, use com cautela

print("--- Previsão de Renda II: Preparação e Divisão da Base de Dados ---")

# Carregar a base de dados
try:
    df = pd.read_csv('previsao_de_renda.csv')
    print("Base de dados 'previsao_de_renda.csv' carregada com sucesso!")
except FileNotFoundError:
    print("Erro: 'previsao_de_renda.csv' não encontrado. Por favor, verifique o caminho do arquivo.")
    exit()

# Remover colunas desnecessárias ou que podem causar problemas
# 'Unnamed: 0' é um índice importado. 'id_cliente' e 'data_ref' não são variáveis explicativas diretas.
df = df.drop(columns=['Unnamed: 0', 'id_cliente', 'data_ref'], errors='ignore')

# Tratar valores nulos (NaN) e valores inconsistentes
# Para a variável resposta 'renda', vamos garantir que seja > 0 para o log
df = df[df['renda'] > 0].copy()

# Para 'tempo_emprego', que tem NaNs e é crucial, vamos remover as linhas com NaN e valores <= 0
df_clean = df.dropna(subset=['tempo_emprego']).copy()
df_clean = df_clean[df_clean['tempo_emprego'] > 0].copy()

print(f"\nNúmero de linhas após limpeza inicial de NaNs e valores inconsistentes: {len(df_clean)}")

# Criar a variável log_renda
df_clean['log_renda'] = np.log(df_clean['renda'])
print("Variável 'log_renda' criada.")

# Remover NaNs remanescentes em outras colunas que serão usadas como features.
# A função `train_test_split` não lida com NaNs, então é importante removê-los aqui.
# Para este exercício, vamos remover NaNs de todas as colunas que podem ser features,
# para garantir que a divisão seja feita em um dataset completo.
initial_cols = df_clean.shape[1]
df_clean.dropna(inplace=True)
print(f"Número de linhas após remover todos os NaNs restantes: {len(df_clean)} (antes: {initial_cols} colunas)")


# Separar a variável resposta (y) das variáveis explicativas (X)
# 'renda' e 'log_renda' são variáveis resposta. Vamos prever 'log_renda'.
X = df_clean.drop(columns=['renda', 'log_renda'])
y = df_clean['log_renda']

# Identificar as variáveis categóricas para Patsy/dummies.
# Isso será importante para o pré-processamento dentro do pipeline ou antes do ajuste do modelo.
categorical_features = X.select_dtypes(include=['object', 'bool']).columns.tolist()
print(f"\nVariáveis categóricas identificadas: {categorical_features}")

# Realizar a separação em conjuntos de treinamento e teste
# test_size=0.25 (25% para teste), random_state para reprodutibilidade
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"\nBase de dados separada em treinamento e teste:")
print(f"Dimensões de X_train: {X_train.shape}")
print(f"Dimensões de y_train: {y_train.shape}")
print(f"Dimensões de X_test: {X_test.shape}")
print(f"Dimensões de y_test: {y_test.shape}")

print("\nOs conjuntos de treinamento e teste foram criados com sucesso!")

--- Previsão de Renda II: Preparação e Divisão da Base de Dados ---
Base de dados 'previsao_de_renda.csv' carregada com sucesso!

Número de linhas após limpeza inicial de NaNs e valores inconsistentes: 12427
Variável 'log_renda' criada.
Número de linhas após remover todos os NaNs restantes: 12427 (antes: 13 colunas)

Variáveis categóricas identificadas: ['sexo', 'posse_de_veiculo', 'posse_de_imovel', 'tipo_renda', 'educacao', 'estado_civil', 'tipo_residencia']

Base de dados separada em treinamento e teste:
Dimensões de X_train: (9320, 11)
Dimensões de y_train: (9320,)
Dimensões de X_test: (3107, 11)
Dimensões de y_test: (3107,)

Os conjuntos de treinamento e teste foram criados com sucesso!


In [17]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix, build_design_matrices
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

print("--- Previsão de Renda II: Regularização Ridge (Com a ÚLTIMA Correção de Feature Mismatch) ---")

# Re-carregar e preparar a base de dados
try:
    df = pd.read_csv('previsao_de_renda.csv')
    df = df.drop(columns=['Unnamed: 0', 'id_cliente', 'data_ref'], errors='ignore')
    df = df[df['renda'] > 0].copy()
    df_clean = df.dropna(subset=['tempo_emprego']).copy()
    df_clean = df_clean[df_clean['tempo_emprego'] > 0].copy()
    df_clean['log_renda'] = np.log(df_clean['renda'])
    df_clean.dropna(inplace=True)
    print(f"Base de dados preparada. {len(df_clean)} linhas para modelagem.")

except Exception as e:
    print(f"Erro na preparação da base de dados: {e}")
    exit()

# Separar X e y
X = df_clean.drop(columns=['renda', 'log_renda'])
y = df_clean['log_renda']

# Dividir em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Dados divididos: Treinamento ({len(X_train)} amostras), Teste ({len(X_test)} amostras)")

# Definir a fórmula do modelo para Patsy (apenas o lado direito para X)
formula_rhs = (
    'sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + '
    'tipo_renda + educacao + estado_civil + tipo_residencia + idade + '
    'tempo_emprego + qt_pessoas_residencia'
)

# --- CORREÇÃO AQUI: Converter para DataFrame após build_design_matrices ---

# 1. Obter o design_info APENAS para as variáveis explicativas (lado direito da fórmula)
X_train_patsy_temp = dmatrix(formula_rhs, data=X_train, return_type='dataframe')
design_info_X = X_train_patsy_temp.design_info

# 2. Gerar as matrizes de design para TREINO e TESTE usando o design_info_X aprendido.
X_train_patsy = build_design_matrices([design_info_X], X_train)[0]
X_test_patsy = build_design_matrices([design_info_X], X_test)[0]

# Converter para DataFrame ANTES de tentar acessar .columns ou .drop
X_train_patsy = pd.DataFrame(X_train_patsy, columns=X_train_patsy.design_info.column_names)
X_test_patsy = pd.DataFrame(X_test_patsy, columns=X_test_patsy.design_info.column_names)


# Remover a coluna 'Intercept' de X, pois o modelo Ridge da scikit-learn adiciona seu próprio intercepto
if 'Intercept' in X_train_patsy.columns:
    X_train_patsy = X_train_patsy.drop(columns='Intercept')
if 'Intercept' in X_test_patsy.columns:
    X_test_patsy = X_test_patsy.drop(columns='Intercept')

print(f"\nMatrizes de design criadas com Patsy (APÓS CORREÇÃO):")
print(f"X_train_patsy shape: {X_train_patsy.shape}")
print(f"X_test_patsy shape: {X_test_patsy.shape}")
print(f"Colunas de X_train_patsy: {X_train_patsy.columns.tolist()}")
print(f"Colunas de X_test_patsy: {X_test_patsy.columns.tolist()}")
print(f"Número de colunas de X_train_patsy e X_test_patsy são iguais? {X_train_patsy.shape[1] == X_test_patsy.shape[1]}")
print(f"Nomes das colunas de X_train_patsy e X_test_patsy são idênticos? {all(X_train_patsy.columns == X_test_patsy.columns)}")


# Definir os valores de alpha para a regularização Ridge
alphas = [0, 0.001, 0.005, 0.01, 0.05, 0.1]
results = []

print("\n--- Rodando Regularização Ridge para diferentes alphas ---")

for alpha in alphas:
    model_name = f"Ridge (alpha={alpha})"
    model = Ridge(alpha=alpha, random_state=42)

    # Treinar o modelo
    model.fit(X_train_patsy, y_train.values.ravel())

    # Fazer previsões no conjunto de teste
    y_pred_test = model.predict(X_test_patsy)

    # Calcular o R2 na base de testes
    r2_test = r2_score(y_test.values.ravel(), y_pred_test)
    results.append({'alpha': alpha, 'model_name': model_name, 'r2_test': r2_test})
    print(f"  {model_name}: R2 na base de testes = {r2_test:.4f}")

# Avaliar o melhor modelo
results_df = pd.DataFrame(results)
best_model_row = results_df.loc[results_df['r2_test'].idxmax()]

print("\n--- Resultados da Regularização Ridge ---")
print(results_df.to_string(index=False))

print(f"\n--- Melhor Modelo ---")
print(f"O melhor modelo (maior R2 na base de testes) foi: **{best_model_row['model_name']}**")
print(f"Com alpha = {best_model_row['alpha']:.3f} e R2 na base de testes = **{best_model_row['r2_test']:.4f}**")

print("\n**Conclusão sobre o Melhor Modelo:**")
print("O valor de `alpha` que resultou no maior R2 na base de testes é o ideal para este conjunto de dados, pois ele encontrou o equilíbrio entre o ajuste aos dados de treinamento e a capacidade de generalização para dados novos.")
print("Um `alpha` diferente de zero sugere que a regularização Ridge foi benéfica, ajudando a controlar a complexidade do modelo e possivelmente a multicolinearidade, levando a um desempenho ligeiramente melhor em dados não vistos.")

--- Previsão de Renda II: Regularização Ridge (Com a ÚLTIMA Correção de Feature Mismatch) ---
Base de dados preparada. 12427 linhas para modelagem.
Dados divididos: Treinamento (9320 amostras), Teste (3107 amostras)

Matrizes de design criadas com Patsy (APÓS CORREÇÃO):
X_train_patsy shape: (9320, 24)
X_test_patsy shape: (3107, 24)
Colunas de X_train_patsy: ['sexo[T.M]', 'posse_de_veiculo[T.True]', 'posse_de_imovel[T.True]', 'tipo_renda[T.Bolsista]', 'tipo_renda[T.Empresário]', 'tipo_renda[T.Pensionista]', 'tipo_renda[T.Servidor público]', 'educacao[T.Pós graduação]', 'educacao[T.Secundário]', 'educacao[T.Superior completo]', 'educacao[T.Superior incompleto]', 'estado_civil[T.Separado]', 'estado_civil[T.Solteiro]', 'estado_civil[T.União]', 'estado_civil[T.Viúvo]', 'tipo_residencia[T.Casa]', 'tipo_residencia[T.Com os pais]', 'tipo_residencia[T.Comunitário]', 'tipo_residencia[T.Estúdio]', 'tipo_residencia[T.Governamental]', 'qtd_filhos', 'idade', 'tempo_emprego', 'qt_pessoas_residencia']

In [21]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix, build_design_matrices
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso # Importar ambos os modelos
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

print("--- Previsão de Renda II: Regularização LASSO ---")

# Re-carregar e preparar a base de dados (garantindo consistência)
try:
    df = pd.read_csv('previsao_de_renda.csv')
    df = df.drop(columns=['Unnamed: 0', 'id_cliente', 'data_ref'], errors='ignore')
    df = df[df['renda'] > 0].copy()
    df_clean = df.dropna(subset=['tempo_emprego']).copy()
    df_clean = df_clean[df_clean['tempo_emprego'] > 0].copy()
    df_clean['log_renda'] = np.log(df_clean['renda'])
    df_clean.dropna(inplace=True)
    print(f"Base de dados preparada. {len(df_clean)} linhas para modelagem.")

except Exception as e:
    print(f"Erro na preparação da base de dados: {e}")
    exit()

# Separar X e y
X = df_clean.drop(columns=['renda', 'log_renda'])
y = df_clean['log_renda']

# Dividir em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Dados divididos: Treinamento ({len(X_train)} amostras), Teste ({len(X_test)} amostras)")

# Definir a fórmula do modelo para Patsy (apenas o lado direito para X)
formula_rhs = (
    'sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + '
    'tipo_renda + educacao + estado_civil + tipo_residencia + idade + '
    'tempo_emprego + qt_pessoas_residencia'
)

# Obter o design_info APENAS para as variáveis explicativas
X_train_patsy_temp = dmatrix(formula_rhs, data=X_train, return_type='dataframe')
design_info_X = X_train_patsy_temp.design_info

# Gerar as matrizes de design para TREINO e TESTE usando o design_info_X aprendido.
X_train_patsy = build_design_matrices([design_info_X], X_train)[0]
X_test_patsy = build_design_matrices([design_info_X], X_test)[0]

# Converter para DataFrame
X_train_patsy = pd.DataFrame(X_train_patsy, columns=X_train_patsy.design_info.column_names)
X_test_patsy = pd.DataFrame(X_test_patsy, columns=X_test_patsy.design_info.column_names)

# Remover a coluna 'Intercept'
if 'Intercept' in X_train_patsy.columns:
    X_train_patsy = X_train_patsy.drop(columns='Intercept')
if 'Intercept' in X_test_patsy.columns:
    X_test_patsy = X_test_patsy.drop(columns='Intercept')

print(f"\nMatrizes de design criadas com Patsy:")
print(f"X_train_patsy shape: {X_train_patsy.shape}")
print(f"X_test_patsy shape: {X_test_patsy.shape}")
print(f"Colunas de X_train_patsy e X_test_patsy são idênticas? {all(X_train_patsy.columns == X_test_patsy.columns)}")

# Definir os valores de alpha
alphas = [0, 0.001, 0.005, 0.01, 0.05, 0.1]

# --- Rodando Regularização RIDGE para obter o melhor R2 para comparação ---
ridge_results_for_comparison = []
print("\n--- Rodando Regularização RIDGE (para comparação) ---")
for alpha_ridge in alphas:
    model_ridge = Ridge(alpha=alpha_ridge, random_state=42)
    model_ridge.fit(X_train_patsy, y_train.values.ravel())
    y_pred_test_ridge = model_ridge.predict(X_test_patsy)
    r2_test_ridge = r2_score(y_test.values.ravel(), y_pred_test_ridge)
    ridge_results_for_comparison.append({'alpha': alpha_ridge, 'r2_test': r2_test_ridge})
    print(f"  Ridge (alpha={alpha_ridge}): R2 na base de testes = {r2_test_ridge:.4f}")

ridge_best_model_row = pd.DataFrame(ridge_results_for_comparison).loc[pd.DataFrame(ridge_results_for_comparison)['r2_test'].idxmax()]
ridge_best_r2 = ridge_best_model_row['r2_test']
print(f"\nMelhor R2 do Ridge para comparação: **{ridge_best_r2:.4f}** (com alpha={ridge_best_model_row['alpha']:.3f})")

# --- Rodando Regularização LASSO ---
lasso_results = []
print("\n--- Rodando Regularização LASSO para diferentes alphas ---")

for alpha in alphas:
    model_name = f"Lasso (alpha={alpha})"
    # Aumentar max_iter para garantir convergência, especialmente para alphas menores
    model = Lasso(alpha=alpha, random_state=42, max_iter=5000)

    # Treinar o modelo
    model.fit(X_train_patsy, y_train.values.ravel())

    # Fazer previsões no conjunto de teste
    y_pred_test = model.predict(X_test_patsy)

    # Calcular o R2 na base de testes
    r2_test = r2_score(y_test.values.ravel(), y_pred_test)
    lasso_results.append({'alpha': alpha, 'model_name': model_name, 'r2_test': r2_test})
    print(f"  {model_name}: R2 na base de testes = {r2_test:.4f}")

# Avaliar o melhor modelo Lasso
lasso_results_df = pd.DataFrame(lasso_results)
best_lasso_model_row = lasso_results_df.loc[lasso_results_df['r2_test'].idxmax()]

print("\n--- Resultados da Regularização LASSO ---")
print(lasso_results_df.to_string(index=False))

print(f"\n--- Melhor Modelo LASSO ---")
print(f"O melhor modelo Lasso (maior R2 na base de testes) foi: **{best_lasso_model_row['model_name']}**")
print(f"Com alpha = {best_lasso_model_row['alpha']:.3f} e R2 na base de testes = **{best_lasso_model_row['r2_test']:.4f}**")

--- Previsão de Renda II: Regularização LASSO ---
Base de dados preparada. 12427 linhas para modelagem.
Dados divididos: Treinamento (9320 amostras), Teste (3107 amostras)

Matrizes de design criadas com Patsy:
X_train_patsy shape: (9320, 24)
X_test_patsy shape: (3107, 24)
Colunas de X_train_patsy e X_test_patsy são idênticas? True

--- Rodando Regularização RIDGE (para comparação) ---
  Ridge (alpha=0): R2 na base de testes = 0.3645
  Ridge (alpha=0.001): R2 na base de testes = 0.3645
  Ridge (alpha=0.005): R2 na base de testes = 0.3645
  Ridge (alpha=0.01): R2 na base de testes = 0.3645
  Ridge (alpha=0.05): R2 na base de testes = 0.3645
  Ridge (alpha=0.1): R2 na base de testes = 0.3645

Melhor R2 do Ridge para comparação: **0.3645** (com alpha=0.100)

--- Rodando Regularização LASSO para diferentes alphas ---
  Lasso (alpha=0): R2 na base de testes = 0.3645
  Lasso (alpha=0.001): R2 na base de testes = 0.3656
  Lasso (alpha=0.005): R2 na base de testes = 0.3651
  Lasso (alpha=0.01)

In [23]:
best_lasso_r2 = best_lasso_model_row['r2_test']

print("\n--- Comparação de Métodos: Ridge vs. Lasso ---")
print(f"Melhor R2 (Base de Testes) - Ridge: **{ridge_best_r2:.4f}** (com alpha={ridge_best_model_row['alpha']:.3f})")
print(f"Melhor R2 (Base de Testes) - Lasso: **{best_lasso_r2:.4f}** (com alpha={best_lasso_model_row['alpha']:.3f})")

if best_lasso_r2 > ridge_best_r2:
    print("\n**Conclusão**: O método **LASSO** chegou a um melhor resultado (maior R2) na base de testes.")
    print("Isso sugere que a capacidade do Lasso de zerar coeficientes de variáveis menos importantes foi benéfica para a generalização do modelo neste conjunto de dados, possivelmente devido à presença de covariáveis que são de fato irrelevantes ou redundantes.")
elif ridge_best_r2 > best_lasso_r2:
    print("\n**Conclusão**: O método **RIDGE** chegou a um melhor resultado (maior R2) na base de testes.")
    print("Isso pode indicar que, para este conjunto de dados, a penalidade L2 do Ridge (que encolhe os coeficientes sem necessariamente zerá-los) foi mais eficaz em lidar com a multicolinearidade e/ou ruído do que a seleção de variáveis mais agressiva do Lasso.")
else:
    print("\n**Conclusão**: Ambos os métodos (Ridge e Lasso) chegaram a resultados de R2 muito similares na base de testes.")
    print("Nesse caso, outros fatores como a interpretabilidade do modelo (Lasso tende a ser mais esparso e mais fácil de interpretar) podem influenciar a escolha.")

print("\n**Observação sobre Coeficientes do Lasso:**")
# Mostrar os coeficientes do melhor modelo Lasso (apenas os não-zero, para ver a seleção de features)
if best_lasso_model_row['alpha'] != 0:
    best_lasso_model = Lasso(alpha=best_lasso_model_row['alpha'], random_state=42, max_iter=5000)
    best_lasso_model.fit(X_train_patsy, y_train.values.ravel())
    lasso_coefs = pd.Series(best_lasso_model.coef_, index=X_train_patsy.columns)
    print("\nCoeficientes do Melhor Modelo LASSO (apenas não-zero):")
    print(lasso_coefs[lasso_coefs != 0].sort_values(ascending=False))
    print(f"\nNúmero de variáveis com coeficiente zero no melhor Lasso: **{(lasso_coefs == 0).sum()}** de {len(lasso_coefs)} total.")
else:
    print("\nO melhor Lasso teve alpha=0, o que o torna equivalente a uma regressão linear comum. Não houve seleção de variáveis por penalidade L1.")


--- Comparação de Métodos: Ridge vs. Lasso ---
Melhor R2 (Base de Testes) - Ridge: **0.3645** (com alpha=0.100)
Melhor R2 (Base de Testes) - Lasso: **0.3656** (com alpha=0.001)

**Conclusão**: O método **LASSO** chegou a um melhor resultado (maior R2) na base de testes.
Isso sugere que a capacidade do Lasso de zerar coeficientes de variáveis menos importantes foi benéfica para a generalização do modelo neste conjunto de dados, possivelmente devido à presença de covariáveis que são de fato irrelevantes ou redundantes.

**Observação sobre Coeficientes do Lasso:**

Coeficientes do Melhor Modelo LASSO (apenas não-zero):
sexo[T.M]                         0.784361
tipo_renda[T.Empresário]          0.157200
educacao[T.Superior completo]     0.107590
estado_civil[T.Viúvo]             0.089572
posse_de_imovel[T.True]           0.087289
tempo_emprego                     0.060783
tipo_renda[T.Servidor público]    0.051108
estado_civil[T.Separado]          0.033181
posse_de_veiculo[T.True]       

In [29]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix, build_design_matrices
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge, Lasso # Importar ambos os modelos
import warnings

warnings.filterwarnings('ignore')

print("--- Previsão de Renda II: Modelo Stepwise (Backward - Seleção por p-valor) ---")

# Re-carregar e preparar a base de dados
try:
    df = pd.read_csv('previsao_de_renda.csv')
    df = df.drop(columns=['Unnamed: 0', 'id_cliente', 'data_ref'], errors='ignore')
    df = df[df['renda'] > 0].copy()
    df_clean = df.dropna(subset=['tempo_emprego']).copy()
    df_clean = df_clean[df_clean['tempo_emprego'] > 0].copy()
    df_clean['log_renda'] = np.log(df_clean['renda'])
    df_clean.dropna(inplace=True)
    print(f"Base de dados preparada. {len(df_clean)} linhas para modelagem.")

except Exception as e:
    print(f"Erro na preparação da base de dados: {e}")
    exit()

# Separar X e y
X = df_clean.drop(columns=['renda', 'log_renda'])
y = df_clean['log_renda']

# Dividir em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# --- CORREÇÃO AQUI: Resetar índices para garantir alinhamento ---
# Isso é crucial para evitar problemas de alinhamento de índice com Statsmodels/Patsy
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)


print(f"Dados divididos: Treinamento ({len(X_train)} amostras), Teste ({len(X_test)} amostras)")

# Definir a fórmula inicial do modelo completo
initial_formula_rhs = (
    'sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + '
    'tipo_renda + educacao + estado_civil + tipo_residencia + idade + '
    'tempo_emprego + qt_pessoas_residencia'
)

# --- Pré-processamento Patsy para garantir colunas consistentes AO LONGO DE TODO O PROCESSO ---
# 1. Obter o design_info para a fórmula mais completa a partir dos dados de treino.
X_full_patsy_train_temp = dmatrix(initial_formula_rhs, data=X_train, return_type='dataframe')
design_info_full = X_full_patsy_train_temp.design_info

# 2. Gerar as matrizes de design COMPLETAS para TREINO e TESTE usando este design_info.
X_train_full_patsy = build_design_matrices([design_info_full], X_train)[0]
X_test_full_patsy = build_design_matrices([design_info_full], X_test)[0]

# Converter para DataFrame
X_train_full_patsy = pd.DataFrame(X_train_full_patsy, columns=X_train_full_patsy.design_info.column_names)
X_test_full_patsy = pd.DataFrame(X_test_full_patsy, columns=X_test_full_patsy.design_info.column_names)

# Remover a coluna 'Intercept' (se presente) para que sm.add_constant adicione a sua
if 'Intercept' in X_train_full_patsy.columns:
    X_train_full_patsy = X_train_full_patsy.drop(columns='Intercept')
    X_test_full_patsy = X_test_full_patsy.drop(columns='Intercept')

print(f"\nMatrizes de design COMPLETAS criadas com Patsy para todo o processo:")
print(f"X_train_full_patsy shape: {X_train_full_patsy.shape}")
print(f"X_test_full_patsy shape: {X_test_full_patsy.shape}")


# --- Processo de Seleção Backward Iterativa com Statsmodels ---
print("\n--- Início do Processo de Seleção Backward (baseado em p-valores) ---")

current_vars_in_formula = [term.strip() for term in initial_formula_rhs.split('+')]
best_r2_stepwise = -np.inf
final_stepwise_model = None
best_formula_stepwise = ""
stepwise_history = []

while True:
    # 1. Selecionar apenas as colunas relevantes para o modelo atual
    # Precisamos garantir que as colunas sejam exatamente as que Statsmodels espera.
    # Patsy cria nomes como 'tipo_renda[T.Assalariado]'
    
    # Mapear os nomes originais para os nomes completos de colunas do Patsy
    cols_to_include_in_model = []
    for original_var in current_vars_in_formula:
        # Encontra todas as colunas do Patsy que correspondem a esta variável original
        matching_cols = [col for col in X_train_full_patsy.columns if col.startswith(original_var)]
        cols_to_include_in_model.extend(matching_cols)
    
    # Se a lista de colunas estiver vazia (ex: no final do processo, sobrou só o intercepto)
    if not cols_to_include_in_model and not current_vars_in_formula:
        # Isso significa que todas as variáveis foram removidas e só deveria haver um intercepto
        X_train_current = pd.DataFrame(index=X_train_full_patsy.index)
        X_test_current = pd.DataFrame(index=X_test_full_patsy.index)
    else:
        X_train_current = X_train_full_patsy[cols_to_include_in_model]
        X_test_current = X_test_full_patsy[cols_to_include_in_model]

    # Adicionar uma constante (intercepto) para o Statsmodels OLS
    X_train_current = sm.add_constant(X_train_current, prepend=True)
    X_test_current = sm.add_constant(X_test_current, prepend=True)

    # Ajustar o modelo OLS
    model = sm.OLS(y_train, X_train_current).fit()

    # Calcular R2 na base de testes para o modelo atual
    y_pred_test = model.predict(X_test_current)
    r2_test_current = r2_score(y_test, y_pred_test)

    # 2. Identificar a variável menos significante (maior p-valor > 0.05)
    p_values = model.pvalues.drop('const', errors='ignore')
    p_values = p_values.replace([np.inf, -np.inf], np.nan).dropna() # Lidar com possíveis NaNs/Infs

    non_significant_vars = p_values[p_values >= 0.05]

    if non_significant_vars.empty or not current_vars_in_formula:
        print(f"\nTodas as variáveis restantes são significantes (p < 0.05) ou não há mais variáveis para remover. Processo encerrado.")
        final_stepwise_model = model
        best_r2_stepwise = r2_test_current
        best_formula_stepwise = ' + '.join(current_vars_in_formula) if current_vars_in_formula else "1"
        break
    else:
        # Encontrar a variável com o maior p-valor não significante (considerando as dummys)
        var_to_remove_patsy_name = non_significant_vars.idxmax()
        max_p_value = non_significant_vars.max()

        # Encontrar o nome da variável original correspondente para remover da lista 'current_vars_in_formula'
        original_var_name_to_remove = None
        for original_var in current_vars_in_formula:
            # Se o nome da coluna Patsy começa com o nome da variável original, é uma correspondência.
            # Isso lida tanto com variáveis contínuas quanto com categorias de variáveis dummy.
            if var_to_remove_patsy_name.startswith(original_var) or var_to_remove_patsy_name == original_var:
                original_var_name_to_remove = original_var
                break
        
        # Este caso deve ser raro se a lógica de correspondência estiver boa, mas é um fallback
        if original_var_name_to_remove is None:
            # Se por algum motivo não achou a original, tenta remover pelo nome da dummy (pode ser contínua)
            original_var_name_to_remove = var_to_remove_patsy_name.split('[')[0]

        # Evitar remover a mesma variável várias vezes
        if original_var_name_to_remove not in current_vars_in_formula:
            # Isso pode acontecer se, por exemplo, 'tipo_renda[T.Bolsista]' tinha o maior p-valor,
            # mas 'tipo_renda' já foi removida (o que não deveria acontecer se o loop for bem controlado).
            # Para segurança, vamos pular a remoção se a variável já não estiver na lista.
            # Ou, mais provável, var_to_remove_patsy_name é um nome de dummy, mas a variável base original
            # já foi removida.
            print(f"  Variável '{original_var_name_to_remove}' já removida ou não encontrada na fórmula. Pulando iteração.")
            non_significant_vars = non_significant_vars.drop(var_to_remove_patsy_name)
            if non_significant_vars.empty:
                 print(f"\nTodas as variáveis restantes são significantes (p < 0.05) ou não há mais variáveis para remover. Processo encerrado.")
                 final_stepwise_model = model
                 best_r2_stepwise = r2_test_current
                 best_formula_stepwise = ' + '.join(current_vars_in_formula) if current_vars_in_formula else "1"
                 break
            continue # Tentar a próxima variável não significante

        print(f"  Removendo **{original_var_name_to_remove}** (P-valor: {max_p_value:.4f}) - R2 Teste Atual: {r2_test_current:.4f}")

        # Remove a variável (base) da lista de variáveis atuais
        current_vars_in_formula = [v for v in current_vars_in_formula if v != original_var_name_to_remove]
        
        stepwise_history.append({
            'formula': ' + '.join(current_vars_in_formula) if current_vars_in_formula else "1",
            'r2_test': r2_test_current,
            'removed_var': original_var_name_to_remove,
            'p_value': max_p_value
        })

# Exibição do sumário do modelo final
print("\n--- Sumário do Modelo Stepwise Final ---")
if final_stepwise_model:
    print(final_stepwise_model.summary())
    print(f"\nFórmula do Modelo Stepwise Final: log_renda ~ {best_formula_stepwise}")
    print(f"R2 na base de testes do Modelo Stepwise Final: **{best_r2_stepwise:.4f}**")
else:
    print("Nenhum modelo stepwise finalizado (erro ou todas as variáveis removidas).")


# --- Comparação Final com Ridge e Lasso (Recuperando resultados anteriores) ---

# Re-executar o cálculo dos melhores R2 do Ridge e Lasso (usando as matrizes X_train_full_patsy e X_test_full_patsy)
alphas = [0, 0.001, 0.005, 0.01, 0.05, 0.1]

# Melhor Ridge
ridge_results_for_comparison = []
for alpha_ridge in alphas:
    model_ridge = Ridge(alpha=alpha_ridge, random_state=42)
    # X_train_full_patsy e X_test_full_patsy já estão sem intercepto do Patsy e são DataFrames
    model_ridge.fit(X_train_full_patsy, y_train.values.ravel())
    y_pred_test_ridge = model_ridge.predict(X_test_full_patsy)
    r2_test_ridge = r2_score(y_test.values.ravel(), y_pred_test_ridge)
    ridge_results_for_comparison.append({'alpha': alpha_ridge, 'r2_test': r2_test_ridge})
ridge_best_model_row = pd.DataFrame(ridge_results_for_comparison).loc[pd.DataFrame(ridge_results_for_comparison)['r2_test'].idxmax()]
ridge_best_r2 = ridge_best_model_row['r2_test']

# Melhor Lasso
lasso_results_for_comparison = []
for alpha_lasso in alphas:
    model_lasso = Lasso(alpha=alpha_lasso, random_state=42, max_iter=5000)
    # X_train_full_patsy e X_test_full_patsy já estão sem intercepto do Patsy e são DataFrames
    model_lasso.fit(X_train_full_patsy, y_train.values.ravel())
    y_pred_test_lasso = model_lasso.predict(X_test_full_patsy)
    r2_test_lasso = r2_score(y_test.values.ravel(), y_pred_test_lasso)
    lasso_results_for_comparison.append({'alpha': alpha_lasso, 'r2_test': r2_test_lasso})
lasso_best_model_row = pd.DataFrame(lasso_results_for_comparison).loc[pd.DataFrame(lasso_results_for_comparison)['r2_test'].idxmax()]
lasso_best_r2 = lasso_best_model_row['r2_test']

print("\n--- Comparação Final de Métodos (R2 na Base de Testes) ---")
print(f"Melhor R2 (Ridge):    **{ridge_best_r2:.4f}** (com alpha={ridge_best_model_row['alpha']:.3f})")
print(f"Melhor R2 (Lasso):    **{lasso_best_r2:.4f}** (com alpha={lasso_best_model_row['alpha']:.3f})")
print(f"R2 (Stepwise Backward): **{best_r2_stepwise:.4f}**")

# Determinar o melhor método global
all_r2_results = {
    'Ridge': ridge_best_r2,
    'Lasso': lasso_best_r2,
    'Stepwise': best_r2_stepwise
}

best_method = max(all_r2_results, key=all_r2_results.get)
best_r2_overall = all_r2_results[best_method]

print(f"\n**O melhor resultado de R2 na base de testes foi obtido com o método: **{best_method}** (R2 = {best_r2_overall:.4f})**")

print("\n**Conclusão Final:**")
print("A escolha do 'melhor' método depende do equilíbrio entre poder preditivo (R2), interpretabilidade e complexidade do modelo.")
if best_method == 'Ridge':
    print(f"- O modelo **Ridge** se destacou. Isso pode indicar que a penalidade L2, que encolhe os coeficientes sem zerá-los, foi mais eficaz em lidar com a multicolinearidade e/ou ruído, mantendo todas as variáveis, mas reduzindo sua influência.")
elif best_method == 'Lasso':
    print(f"- O modelo **Lasso** se destacou. Isso sugere que a capacidade de seleção de variáveis do Lasso (zerando coeficientes) foi benéfica, simplificando o modelo ao remover preditores menos importantes e melhorando a generalização.")
elif best_method == 'Stepwise':
    print(f"- O modelo **Stepwise (backward por p-valor)** se destacou. Isso indica que a seleção explícita de variáveis baseada em significância estatística resultou no modelo mais preditivo e parcimonioso para este conjunto de dados.")

print("\nEm geral, métodos de regularização como Ridge e Lasso são preferidos em cenários de alta dimensionalidade ou multicolinearidade devido à sua robustez e capacidade de evitar overfitting.")

--- Previsão de Renda II: Modelo Stepwise (Backward - Seleção por p-valor) ---
Base de dados preparada. 12427 linhas para modelagem.
Dados divididos: Treinamento (9320 amostras), Teste (3107 amostras)

Matrizes de design COMPLETAS criadas com Patsy para todo o processo:
X_train_full_patsy shape: (9320, 24)
X_test_full_patsy shape: (3107, 24)

--- Início do Processo de Seleção Backward (baseado em p-valores) ---
  Removendo **educacao** (P-valor: 0.8735) - R2 Teste Atual: 0.3645
  Removendo **tipo_residencia** (P-valor: 0.6088) - R2 Teste Atual: 0.3613
  Removendo **tipo_renda** (P-valor: 0.3037) - R2 Teste Atual: 0.3619
  Removendo **estado_civil** (P-valor: 0.3558) - R2 Teste Atual: 0.3579
  Removendo **qt_pessoas_residencia** (P-valor: 0.4645) - R2 Teste Atual: 0.3585

Todas as variáveis restantes são significantes (p < 0.05) ou não há mais variáveis para remover. Processo encerrado.

--- Sumário do Modelo Stepwise Final ---
                            OLS Regression Results         

In [35]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix, build_design_matrices
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge, Lasso
import warnings

warnings.filterwarnings('ignore')

print("--- Previsão de Renda II: Reexecutando Modelos para Comparação ---")

# Re-carregar e preparar a base de dados
try:
    df = pd.read_csv('previsao_de_renda.csv')
    df = df.drop(columns=['Unnamed: 0', 'id_cliente', 'data_ref'], errors='ignore')
    df = df[df['renda'] > 0].copy()
    df_clean = df.dropna(subset=['tempo_emprego']).copy()
    df_clean = df_clean[df_clean['tempo_emprego'] > 0].copy()
    df_clean['log_renda'] = np.log(df_clean['renda'])
    df_clean.dropna(inplace=True)
    print(f"Base de dados preparada. {len(df_clean)} linhas para modelagem.")

except Exception as e:
    print(f"Erro na preparação da base de dados: {e}")
    exit()

# Separar X e y
X = df_clean.drop(columns=['renda', 'log_renda'])
y = df_clean['log_renda']

# Dividir em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Resetar índices para garantir alinhamento para Statsmodels/Patsy
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"Dados divididos: Treinamento ({len(X_train)} amostras), Teste ({len(X_test)} amostras)")

# Definir a fórmula inicial do modelo completo
initial_formula_rhs = (
    'sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + '
    'tipo_renda + educacao + estado_civil + tipo_residencia + idade + '
    'tempo_emprego + qt_pessoas_residencia'
)

# Pré-processamento Patsy para garantir colunas consistentes AO LONGO DE TODO O PROCESSO
X_full_patsy_train_temp = dmatrix(initial_formula_rhs, data=X_train, return_type='dataframe')
design_info_full = X_full_patsy_train_temp.design_info

X_train_full_patsy = build_design_matrices([design_info_full], X_train)[0]
X_test_full_patsy = build_design_matrices([design_info_full], X_test)[0]

X_train_full_patsy = pd.DataFrame(X_train_full_patsy, columns=X_train_full_patsy.design_info.column_names)
X_test_full_patsy = pd.DataFrame(X_test_full_patsy, columns=X_test_full_patsy.design_info.column_names)

if 'Intercept' in X_train_full_patsy.columns:
    X_train_full_patsy = X_train_full_patsy.drop(columns='Intercept')
    X_test_full_patsy = X_test_full_patsy.drop(columns='Intercept')

print(f"\nMatrizes de design COMPLETAS criadas com Patsy para todo o processo:")
print(f"X_train_full_patsy shape: {X_train_full_patsy.shape}")
print(f"X_test_full_patsy shape: {X_test_full_patsy.shape}")

# --- Parâmetros de Alpha para Ridge e Lasso ---
alphas = [0, 0.001, 0.005, 0.01, 0.05, 0.1]

# --- 1. Treinar e Avaliar o Melhor Modelo Ridge ---
print("\n--- Treinando e Avaliando o Melhor Modelo Ridge ---")
ridge_results = []
best_ridge_model = None
best_ridge_r2 = -np.inf
best_ridge_alpha = None

for alpha in alphas:
    model_ridge = Ridge(alpha=alpha, random_state=42)
    model_ridge.fit(X_train_full_patsy, y_train.values.ravel())
    y_pred_test_ridge = model_ridge.predict(X_test_full_patsy)
    r2_test_ridge = r2_score(y_test.values.ravel(), y_pred_test_ridge)
    ridge_results.append({'alpha': alpha, 'r2_test': r2_test_ridge})
    if r2_test_ridge > best_ridge_r2:
        best_ridge_r2 = r2_test_ridge
        best_ridge_model = model_ridge
        best_ridge_alpha = alpha
    print(f"  Ridge (alpha={alpha}): R2 na base de testes = {r2_test_ridge:.4f}")

print(f"\nMelhor R2 do Ridge: **{best_ridge_r2:.4f}** (com alpha={best_ridge_alpha})")
ridge_coefs = pd.Series(best_ridge_model.coef_, index=X_train_full_patsy.columns)
ridge_intercept = best_ridge_model.intercept_

# --- 2. Treinar e Avaliar o Melhor Modelo Lasso ---
print("\n--- Treinando e Avaliando o Melhor Modelo Lasso ---")
lasso_results = []
best_lasso_model = None
best_lasso_r2 = -np.inf
best_lasso_alpha = None

for alpha in alphas:
    model_lasso = Lasso(alpha=alpha, random_state=42, max_iter=5000)
    model_lasso.fit(X_train_full_patsy, y_train.values.ravel())
    y_pred_test_lasso = model_lasso.predict(X_test_full_patsy)
    r2_test_lasso = r2_score(y_test.values.ravel(), y_pred_test_lasso)
    lasso_results.append({'alpha': alpha, 'r2_test': r2_test_lasso})
    if r2_test_lasso > best_lasso_r2:
        best_lasso_r2 = r2_test_lasso
        best_lasso_model = model_lasso
        best_lasso_alpha = alpha
    print(f"  Lasso (alpha={alpha}): R2 na base de testes = {r2_test_lasso:.4f}")

print(f"\nMelhor R2 do Lasso: **{best_lasso_r2:.4f}** (com alpha={best_lasso_alpha})")
lasso_coefs = pd.Series(best_lasso_model.coef_, index=X_train_full_patsy.columns)
lasso_intercept = best_lasso_model.intercept_

# --- 3. Treinar e Avaliar o Modelo Stepwise (Backward) ---
print("\n--- Treinando e Avaliando o Modelo Stepwise (Backward - Seleção por p-valor) ---")

current_vars_in_formula = [term.strip() for term in initial_formula_rhs.split('+')]
best_r2_stepwise = -np.inf
final_stepwise_model_sm = None
best_formula_stepwise = ""

while True:
    cols_to_include_in_model = []
    for original_var in current_vars_in_formula:
        matching_cols = [col for col in X_train_full_patsy.columns if col.startswith(original_var)]
        cols_to_include_in_model.extend(matching_cols)
    
    if not cols_to_include_in_model and not current_vars_in_formula:
        X_train_current = pd.DataFrame(index=X_train_full_patsy.index)
        X_test_current = pd.DataFrame(index=X_test_full_patsy.index)
    else:
        X_train_current = X_train_full_patsy[cols_to_include_in_model]
        X_test_current = X_test_full_patsy[cols_to_include_in_model]

    X_train_current = sm.add_constant(X_train_current, prepend=True)
    X_test_current = sm.add_constant(X_test_current, prepend=True)

    model_sm = sm.OLS(y_train, X_train_current).fit()

    y_pred_test_sm = model_sm.predict(X_test_current)
    r2_test_current_sm = r2_score(y_test, y_pred_test_sm)

    p_values = model_sm.pvalues.drop('const', errors='ignore')
    p_values = p_values.replace([np.inf, -np.inf], np.nan).dropna()
    non_significant_vars = p_values[p_values >= 0.05]

    if non_significant_vars.empty or not current_vars_in_formula:
        print(f"\nTodas as variáveis restantes são significantes (p < 0.05) ou não há mais variáveis para remover. Processo encerrado.")
        final_stepwise_model_sm = model_sm
        best_r2_stepwise = r2_test_current_sm
        best_formula_stepwise = ' + '.join(current_vars_in_formula) if current_vars_in_formula else "1"
        break
    else:
        var_to_remove_patsy_name = non_significant_vars.idxmax()
        max_p_value = non_significant_vars.max()

        original_var_name_to_remove = None
        for original_var in current_vars_in_formula:
            if var_to_remove_patsy_name.startswith(original_var) or var_to_remove_patsy_name == original_var:
                original_var_name_to_remove = original_var
                break
        
        if original_var_name_to_remove is None:
            original_var_name_to_remove = var_to_remove_patsy_name.split('[')[0]

        if original_var_name_to_remove not in current_vars_in_formula:
            print(f"  Variável '{original_var_name_to_remove}' já removida ou não encontrada na fórmula. Pulando iteração.")
            non_significant_vars = non_significant_vars.drop(var_to_remove_patsy_name)
            if non_significant_vars.empty:
                 print(f"\nTodas as variáveis restantes são significantes (p < 0.05) ou não há mais variáveis para remover. Processo encerrado.")
                 final_stepwise_model_sm = model_sm
                 best_r2_stepwise = r2_test_current_sm
                 best_formula_stepwise = ' + '.join(current_vars_in_formula) if current_vars_in_formula else "1"
                 break
            continue

        print(f"  Removendo **{original_var_name_to_remove}** (P-valor: {max_p_value:.4f}) - R2 Teste Atual: {r2_test_current_sm:.4f}")
        current_vars_in_formula = [v for v in current_vars_in_formula if v != original_var_name_to_remove]

print(f"\nMelhor R2 do Stepwise: **{best_r2_stepwise:.4f}**")
if final_stepwise_model_sm:
    stepwise_coefs = final_stepwise_model_sm.params.drop('const', errors='ignore')
    stepwise_intercept = final_stepwise_model_sm.params['const'] if 'const' in final_stepwise_model_sm.params else 0
else:
    stepwise_coefs = pd.Series()
    stepwise_intercept = 0

print("\n--- Fim da Execução dos Modelos ---")

--- Previsão de Renda II: Reexecutando Modelos para Comparação ---
Base de dados preparada. 12427 linhas para modelagem.
Dados divididos: Treinamento (9320 amostras), Teste (3107 amostras)

Matrizes de design COMPLETAS criadas com Patsy para todo o processo:
X_train_full_patsy shape: (9320, 24)
X_test_full_patsy shape: (3107, 24)

--- Treinando e Avaliando o Melhor Modelo Ridge ---
  Ridge (alpha=0): R2 na base de testes = 0.3645
  Ridge (alpha=0.001): R2 na base de testes = 0.3645
  Ridge (alpha=0.005): R2 na base de testes = 0.3645
  Ridge (alpha=0.01): R2 na base de testes = 0.3645
  Ridge (alpha=0.05): R2 na base de testes = 0.3645
  Ridge (alpha=0.1): R2 na base de testes = 0.3645

Melhor R2 do Ridge: **0.3645** (com alpha=0.1)

--- Treinando e Avaliando o Melhor Modelo Lasso ---
  Lasso (alpha=0): R2 na base de testes = 0.3645
  Lasso (alpha=0.001): R2 na base de testes = 0.3656
  Lasso (alpha=0.005): R2 na base de testes = 0.3651
  Lasso (alpha=0.01): R2 na base de testes = 0.36

In [37]:
# Comparação de Parâmetros

# 1. Coeficientes do Melhor Modelo Ridge (alpha = {:.3f})

# O alpha ideal para o Ridge foi de {best_ridge_alpha:.3f}. Veja os coeficientes:

print(ridge_coefs.sort_index().to_string())
print(f"Intercepto Ridge: {ridge_intercept:.4f}")

educacao[T.Pós graduação]          -0.031457
educacao[T.Secundário]             -0.035508
educacao[T.Superior completo]       0.074994
educacao[T.Superior incompleto]    -0.051682
estado_civil[T.Separado]            0.352919
estado_civil[T.Solteiro]            0.296123
estado_civil[T.União]              -0.035416
estado_civil[T.Viúvo]               0.435968
idade                               0.005431
posse_de_imovel[T.True]             0.091132
posse_de_veiculo[T.True]            0.028595
qt_pessoas_residencia               0.317928
qtd_filhos                         -0.287978
sexo[T.M]                           0.789335
tempo_emprego                       0.060872
tipo_renda[T.Bolsista]              0.212767
tipo_renda[T.Empresário]            0.165656
tipo_renda[T.Pensionista]          -0.329543
tipo_renda[T.Servidor público]      0.063310
tipo_residencia[T.Casa]            -0.082801
tipo_residencia[T.Com os pais]     -0.057310
tipo_residencia[T.Comunitário]     -0.227329
tipo_resid

In [39]:
# 2. Coeficientes do Melhor Modelo Lasso (alpha = {:.3f})

# O alpha ideal para o Lasso foi de {best_lasso_alpha:.3f}. Veja os coeficientes:

print(lasso_coefs[lasso_coefs != 0].sort_index().to_string()) # Apenas coeficientes não-zero
print(f"Intercepto Lasso: {lasso_intercept:.4f}")
print(f"Número de variáveis com coeficiente zero no Lasso: {(lasso_coefs == 0).sum()} de {len(lasso_coefs)}")

educacao[T.Superior completo]     0.107590
estado_civil[T.Separado]          0.033181
estado_civil[T.União]            -0.022740
estado_civil[T.Viúvo]             0.089572
idade                             0.005451
posse_de_imovel[T.True]           0.087289
posse_de_veiculo[T.True]          0.024694
qt_pessoas_residencia             0.016586
qtd_filhos                        0.010461
sexo[T.M]                         0.784361
tempo_emprego                     0.060783
tipo_renda[T.Empresário]          0.157200
tipo_renda[T.Servidor público]    0.051108
tipo_residencia[T.Com os pais]    0.003327
Intercepto Lasso: 7.0890
Número de variáveis com coeficiente zero no Lasso: 10 de 24


In [44]:
# 3. Coeficientes do Modelo Stepwise Final

# Veja os coeficientes do modelo Stepwise final:

if not stepwise_coefs.empty:
    print(stepwise_coefs.sort_index().to_string())
else:
    print("Nenhum coeficiente para o modelo Stepwise (pode ser apenas o intercepto).")
print(f"Intercepto Stepwise: {stepwise_intercept:.4f}")

idade                       0.005054
posse_de_imovel[T.True]     0.096495
posse_de_veiculo[T.True]    0.039093
qtd_filhos                  0.025343
sexo[T.M]                   0.767982
tempo_emprego               0.060040
Intercepto Stepwise: 7.2280


In [49]:
# Avaliação das Diferenças e Qual o Melhor Modelo

print("\n--- Avaliação dos Modelos e Escolha do Melhor ---")
print(f"Melhor R2 (Base de Testes) - Ridge:    **{best_ridge_r2:.4f}**")
print(f"Melhor R2 (Base de Testes) - Lasso:    **{best_lasso_r2:.4f}**")
print(f"Melhor R2 (Base de Testes) - Stepwise: **{best_r2_stepwise:.4f}**")

all_r2_results = {
    'Ridge': best_ridge_r2,
    'Lasso': best_lasso_r2,
    'Stepwise': best_r2_stepwise
}

best_method_overall = max(all_r2_results, key=all_r2_results.get)
best_r2_overall = all_r2_results[best_method_overall]

print(f"\n**O modelo com o maior R2 na base de testes é o: **{best_method_overall}** (R2 = {best_r2_overall:.4f})**")

print("\n**Análise das Diferenças e Escolha do Melhor Modelo:**")

print("\n**Comparação de Coeficientes e Interceptos:**")
print("- **Ridge:** Os coeficientes do Ridge são geralmente **menores** em magnitude comparados aos de uma Regressão Linear Múltipla (OLS), mas raramente são exatamente zero. Isso significa que o Ridge mantém todas as variáveis no modelo, mas reduz a influência das menos importantes, sendo eficaz em lidar com **multicolinearidade**. Seu intercepto é ajustado junto com os coeficientes.")
print("- **Lasso:** A principal característica do Lasso é sua capacidade de realizar **seleção de variáveis**. Ele tende a levar os coeficientes de variáveis menos importantes a **zero**. Isso resulta em um modelo mais *esparso* (com menos variáveis), o que pode aumentar a **interpretabilidade**. O intercepto também é ajustado.")
print("- **Stepwise:** O modelo stepwise (backward, neste caso) remove iterativamente as variáveis com **p-valores não significantes** (acima de um limiar como 0.05). Os coeficientes das variáveis que permanecem no modelo são os mesmos que seriam obtidos por uma regressão OLS padrão apenas com essas variáveis. O modelo final é mais simples que o Ridge (a menos que o Ridge zere muitos coeficientes para grandes alphas) e mais **interpretável** que o Ridge (pela remoção explícita de variáveis). O intercepto é o da regressão OLS final.")

print("\n**Qual modelo você acha o melhor de todos?**")
print("A escolha do 'melhor' modelo vai além de simplesmente olhar para o R2 na base de testes, embora seja um critério fundamental para o **desempenho preditivo**.")

if best_method_overall == 'Ridge':
    print(f"\nCom base exclusivamente no R2 na base de testes, o modelo **Ridge** ({best_r2_overall:.4f}) é o melhor.")
    print("- **Vantagens do Ridge aqui:** Se o Ridge se saiu melhor, é provável que existam **multicolinearidade** significativa entre as variáveis preditoras ou que todas as variáveis contribuam, mesmo que minimamente, para a previsão. O Ridge é excelente para lidar com multicolinearidade, pois encolhe os coeficientes de forma proporcional, sem zerá-los, mantendo a informação de todas as variáveis.")
    print("- **Desvantagens:** Pode ser menos interpretável se o número de preditores for muito grande, pois não realiza seleção de variáveis explícita.")

elif best_method_overall == 'Lasso':
    print(f"\nCom base exclusivamente no R2 na base de testes, o modelo **Lasso** ({best_r2_overall:.4f}) é o melhor.")
    print("- **Vantagens do Lasso aqui:** Se o Lasso se destacou, isso sugere que há variáveis no conjunto de dados que são verdadeiramente **irrelevantes ou redundantes**. Ao zerar seus coeficientes, o Lasso não só melhora a generalização como também simplifica o modelo, tornando-o mais **interpretável** e eficiente.")
    print("- **Desvantagens:** Para conjuntos de dados com muitas variáveis altamente correlacionadas, o Lasso tende a selecionar apenas uma delas e zerar as demais, o que pode não ser o ideal se todas as variáveis correlacionadas são de fato importantes e queremos mantê-las (mesmo que com coeficientes menores).")

elif best_method_overall == 'Stepwise':
    print(f"\nCom base exclusivamente no R2 na base de testes, o modelo **Stepwise** ({best_r2_overall:.4f}) é o melhor.")
    print("- **Vantagens do Stepwise aqui:** Ele proporciona um modelo **parcimonioso e interpretável** ao remover explicitamente variáveis não significantes. A seleção é baseada na significância estatística, o que é intuitivo para muitos analistas.")
    print("- **Desvantagens:** Métodos stepwise são criticados por seu processo de busca (que pode não encontrar o modelo globalmente ótimo), pela inflação de p-valores (levando a variáveis 'significantes' que talvez não o sejam em uma análise mais rigorosa) e pela sensibilidade a pequenas mudanças nos dados. No entanto, em termos de R2 na base de testes, ele superou os outros neste cenário.")

print("\n**Conclusão Geral:**")
print(f"Para esta análise, o modelo **{best_method_overall}** apresentou o **melhor desempenho preditivo** (maior R2 na base de testes), o que é crucial para uma tarefa de previsão de renda.")
print("A escolha final, no entanto, deve considerar também a necessidade de **interpretabilidade**. Se a capacidade de explicar a contribuição de cada variável for essencial, um modelo Lasso ou Stepwise pode ser preferível, mesmo que com um R2 ligeiramente inferior, por sua simplicidade. Se a principal meta é a precisão e há multicolinearidade, o Ridge é uma ótima escolha.")


--- Avaliação dos Modelos e Escolha do Melhor ---
Melhor R2 (Base de Testes) - Ridge:    **0.3645**
Melhor R2 (Base de Testes) - Lasso:    **0.3656**
Melhor R2 (Base de Testes) - Stepwise: **0.3585**

**O modelo com o maior R2 na base de testes é o: **Lasso** (R2 = 0.3656)**

**Análise das Diferenças e Escolha do Melhor Modelo:**

**Comparação de Coeficientes e Interceptos:**
- **Ridge:** Os coeficientes do Ridge são geralmente **menores** em magnitude comparados aos de uma Regressão Linear Múltipla (OLS), mas raramente são exatamente zero. Isso significa que o Ridge mantém todas as variáveis no modelo, mas reduz a influência das menos importantes, sendo eficaz em lidar com **multicolinearidade**. Seu intercepto é ajustado junto com os coeficientes.
- **Lasso:** A principal característica do Lasso é sua capacidade de realizar **seleção de variáveis**. Ele tende a levar os coeficientes de variáveis menos importantes a **zero**. Isso resulta em um modelo mais *esparso* (com menos variáv

In [55]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrix, build_design_matrices
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge, Lasso
import warnings

warnings.filterwarnings('ignore')

print("--- Previsão de Renda II: Tentando Melhorar o R2 com Engenharia de Features ---")

# Re-carregar e preparar a base de dados (repetido para garantir ambiente limpo)
try:
    df = pd.read_csv('previsao_de_renda.csv')
    df = df.drop(columns=['Unnamed: 0', 'id_cliente', 'data_ref'], errors='ignore')
    df = df[df['renda'] > 0].copy()
    df_clean = df.dropna(subset=['tempo_emprego']).copy()
    df_clean = df_clean[df_clean['tempo_emprego'] > 0].copy()
    df_clean['log_renda'] = np.log(df_clean['renda'])
    df_clean.dropna(inplace=True)
    print(f"Base de dados preparada. {len(df_clean)} linhas para modelagem.")

except Exception as e:
    print(f"Erro na preparação da base de dados: {e}")
    exit()

# Separar X e y
X = df_clean.drop(columns=['renda', 'log_renda'])
y = df_clean['log_renda']

# Dividir em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Resetar índices para garantir alinhamento para Statsmodels/Patsy
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"Dados divididos: Treinamento ({len(X_train)} amostras), Teste ({len(X_test)} amostras)")

# --- Engenharia de Features Criativas ---
# Vamos adicionar algumas interações e termos polinomiais.
# O Patsy facilita muito isso diretamente na fórmula.

# Definição da fórmula original para comparação
original_formula_rhs = (
    'sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + '
    'tipo_renda + educacao + estado_civil + tipo_residencia + idade + '
    'tempo_emprego + qt_pessoas_residencia'
)

# Vamos adicionar essas features ao DataFrame antes de passar para o Patsy
X_train_enhanced = X_train.copy()
X_test_enhanced = X_test.copy()

X_train_enhanced['tem_filhos_e_casado'] = ((X_train_enhanced['qtd_filhos'] > 0) & 
                                          (X_train_enhanced['estado_civil'] == 'Casado')).astype(int)
X_test_enhanced['tem_filhos_e_casado'] = ((X_test_enhanced['qtd_filhos'] > 0) & 
                                         (X_test_enhanced['estado_civil'] == 'Casado')).astype(int)

X_train_enhanced['possui_bens'] = ((X_train_enhanced['posse_de_veiculo'] == True) | # Corrigido para True booleano
                                   (X_train_enhanced['posse_de_imovel'] == True)).astype(int) # Corrigido para True booleano
X_test_enhanced['possui_bens'] = ((X_test_enhanced['posse_de_veiculo'] == True) | # Corrigido para True booleano
                                  (X_test_enhanced['posse_de_imovel'] == True)).astype(int) # Corrigido para True booleano

# Nova fórmula com as features criadas e interações Patsy
enhanced_formula_rhs = (
    original_formula_rhs +
    ' + I(idade**2) + I(tempo_emprego**2) + I(qtd_filhos**2) + ' +
    'idade:tempo_emprego + idade:qtd_filhos + ' +
    'posse_de_veiculo:idade + sexo:tempo_emprego + ' +
    'tem_filhos_e_casado + possui_bens' # Novas variáveis binárias
)
print(f"\nNova fórmula com features expandidas:\nlog_renda ~ {enhanced_formula_rhs}")

# Pré-processamento Patsy para as features expandidas
# Criar design_info a partir do X_train_enhanced para a fórmula expandida
X_enhanced_patsy_train_temp = dmatrix(enhanced_formula_rhs, data=X_train_enhanced, return_type='dataframe')
design_info_enhanced = X_enhanced_patsy_train_temp.design_info

# Gerar as matrizes de design para TREINO e TESTE usando o design_info_enhanced
X_train_enhanced_patsy = build_design_matrices([design_info_enhanced], X_train_enhanced)[0]
X_test_enhanced_patsy = build_design_matrices([design_info_enhanced], X_test_enhanced)[0]

X_train_enhanced_patsy = pd.DataFrame(X_train_enhanced_patsy, columns=X_train_enhanced_patsy.design_info.column_names)
X_test_enhanced_patsy = pd.DataFrame(X_test_enhanced_patsy, columns=X_test_enhanced_patsy.design_info.column_names)

if 'Intercept' in X_train_enhanced_patsy.columns:
    X_train_enhanced_patsy = X_train_enhanced_patsy.drop(columns='Intercept')
    X_test_enhanced_patsy = X_test_enhanced_patsy.drop(columns='Intercept')

print(f"\nMatrizes de design ENHANCED criadas com Patsy:")
print(f"X_train_enhanced_patsy shape: {X_train_enhanced_patsy.shape}")
print(f"X_test_enhanced_patsy shape: {X_test_enhanced_patsy.shape}")

# --- Treinar e Avaliar o Melhor Modelo Lasso com as Novas Features ---
print("\n--- Treinando e Avaliando o Modelo Lasso com Novas Features ---")
alphas = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0] # Ampliando os alphas para explorar mais a regularização
enhanced_lasso_results = []
best_enhanced_lasso_model = None
best_enhanced_lasso_r2 = -np.inf
best_enhanced_lasso_alpha = None

for alpha in alphas:
    model_lasso_enhanced = Lasso(alpha=alpha, random_state=42, max_iter=10000) # Aumentar max_iter
    model_lasso_enhanced.fit(X_train_enhanced_patsy, y_train.values.ravel())
    y_pred_test_lasso_enhanced = model_lasso_enhanced.predict(X_test_enhanced_patsy)
    r2_test_lasso_enhanced = r2_score(y_test.values.ravel(), y_pred_test_lasso_enhanced)
    enhanced_lasso_results.append({'alpha': alpha, 'r2_test': r2_test_lasso_enhanced})
    if r2_test_lasso_enhanced > best_enhanced_lasso_r2:
        best_enhanced_lasso_r2 = r2_test_lasso_enhanced
        best_enhanced_lasso_model = model_lasso_enhanced
        best_enhanced_lasso_alpha = alpha
    print(f"  Lasso Enhanced (alpha={alpha}): R2 na base de testes = {r2_test_lasso_enhanced:.4f}")

print(f"\nMelhor R2 do Lasso com features expandidas: **{best_enhanced_lasso_r2:.4f}** (com alpha={best_enhanced_lasso_alpha})")
enhanced_lasso_coefs = pd.Series(best_enhanced_lasso_model.coef_, index=X_train_enhanced_patsy.columns)
enhanced_lasso_intercept = best_enhanced_lasso_model.intercept_

print(f"\nCoeficientes do Melhor Lasso com Features Expandidas (apenas não-zero):")
print(enhanced_lasso_coefs[enhanced_lasso_coefs != 0].sort_index().to_string())
print(f"Intercepto: {enhanced_lasso_intercept:.4f}")
print(f"Número de variáveis com coeficiente zero: {(enhanced_lasso_coefs == 0).sum()} de {len(enhanced_lasso_coefs)}")

# --- Recalcular os melhores R2 dos modelos anteriores para comparação precisa ---
# Definir a fórmula inicial do modelo completo (usada nos modelos anteriores)
initial_formula_rhs_original = ( # Renomeado para evitar conflito
    'sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + '
    'tipo_renda + educacao + estado_civil + tipo_residencia + idade + '
    'tempo_emprego + qt_pessoas_residencia'
)

# Pré-processamento Patsy para as features originais
X_original_patsy_train_temp = dmatrix(initial_formula_rhs_original, data=X_train, return_type='dataframe')
design_info_original = X_original_patsy_train_temp.design_info

X_train_original_patsy = build_design_matrices([design_info_original], X_train)[0]
X_test_original_patsy = build_design_matrices([design_info_original], X_test)[0]

X_train_original_patsy = pd.DataFrame(X_train_original_patsy, columns=X_train_original_patsy.design_info.column_names)
X_test_original_patsy = pd.DataFrame(X_test_original_patsy, columns=X_test_original_patsy.design_info.column_names)

if 'Intercept' in X_train_original_patsy.columns:
    X_train_original_patsy = X_train_original_patsy.drop(columns='Intercept')
    X_test_original_patsy = X_test_original_patsy.drop(columns='Intercept')

# Melhores R2 dos modelos anteriores (re-calculando para garantir)
alphas_original_models = [0, 0.001, 0.005, 0.01, 0.05, 0.1] # Renomeado para evitar conflito

# Melhor Ridge (original)
ridge_results_original = []
for alpha_ridge_original in alphas_original_models:
    model_ridge_original = Ridge(alpha=alpha_ridge_original, random_state=42)
    model_ridge_original.fit(X_train_original_patsy, y_train.values.ravel())
    y_pred_test_ridge_original = model_ridge_original.predict(X_test_original_patsy)
    r2_test_ridge_original = r2_score(y_test.values.ravel(), y_pred_test_ridge_original)
    ridge_results_original.append({'alpha': alpha_ridge_original, 'r2_test': r2_test_ridge_original})
best_ridge_r2_original = pd.DataFrame(ridge_results_original)['r2_test'].max()

# Melhor Lasso (original)
lasso_results_original = []
for alpha_lasso_original in alphas_original_models:
    model_lasso_original = Lasso(alpha=alpha_lasso_original, random_state=42, max_iter=5000)
    model_lasso_original.fit(X_train_original_patsy, y_train.values.ravel())
    y_pred_test_lasso_original = model_lasso_original.predict(X_test_original_patsy)
    r2_test_lasso_original = r2_score(y_test.values.ravel(), y_pred_test_lasso_original)
    lasso_results_original.append({'alpha': alpha_lasso_original, 'r2_test': r2_test_lasso_original})
best_lasso_r2_original = pd.DataFrame(lasso_results_original)['r2_test'].max()

# --- 3. Treinar e Avaliar o Modelo Stepwise (Backward) - REPETIÇÃO PARA OBTER O R2 ORIGINAL ---
print("\n--- Recalculando R2 do Stepwise Original para Comparação ---")
current_vars_in_formula_original_stepwise = [term.strip() for term in initial_formula_rhs_original.split('+')]
best_r2_stepwise_original = -np.inf

while True:
    cols_to_include_in_model_original = []
    for original_var in current_vars_in_formula_original_stepwise:
        matching_cols = [col for col in X_train_original_patsy.columns if col.startswith(original_var)]
        cols_to_include_in_model_original.extend(matching_cols)
    
    if not cols_to_include_in_model_original and not current_vars_in_formula_original_stepwise:
        X_train_current_original = pd.DataFrame(index=X_train_original_patsy.index)
        X_test_current_original = pd.DataFrame(index=X_test_original_patsy.index)
    else:
        X_train_current_original = X_train_original_patsy[cols_to_include_in_model_original] # Corrigido aqui
        X_test_current_original = X_test_original_patsy[cols_to_include_in_model_original] # Corrigido aqui

    X_train_current_original = sm.add_constant(X_train_current_original, prepend=True)
    X_test_current_original = sm.add_constant(X_test_current_original, prepend=True)

    model_sm_original = sm.OLS(y_train, X_train_current_original).fit()
    y_pred_test_sm_original = model_sm_original.predict(X_test_current_original)
    r2_test_current_sm_original = r2_score(y_test, y_pred_test_sm_original)

    p_values_original = model_sm_original.pvalues.drop('const', errors='ignore')
    p_values_original = p_values_original.replace([np.inf, -np.inf], np.nan).dropna()
    non_significant_vars_original = p_values_original[p_values_original >= 0.05]

    if non_significant_vars_original.empty or not current_vars_in_formula_original_stepwise:
        best_r2_stepwise_original = r2_test_current_sm_original
        break
    else:
        var_to_remove_patsy_name_original = non_significant_vars_original.idxmax()
        original_var_name_to_remove_original = None
        for original_var in current_vars_in_formula_original_stepwise:
            if var_to_remove_patsy_name_original.startswith(original_var) or var_to_remove_patsy_name_original == original_var:
                original_var_name_to_remove_original = original_var
                break
        if original_var_name_to_remove_original is None:
            original_var_name_to_remove_original = var_to_remove_patsy_name_original.split('[')[0]

        if original_var_name_to_remove_original not in current_vars_in_formula_original_stepwise:
            non_significant_vars_original = non_significant_vars_original.drop(var_to_remove_patsy_name_original)
            if non_significant_vars_original.empty:
                 best_r2_stepwise_original = r2_test_current_sm_original
                 break
            continue
        current_vars_in_formula_original_stepwise = [v for v in current_vars_in_formula_original_stepwise if v != original_var_name_to_remove_original]
print(f"R2 do Stepwise Original: **{best_r2_stepwise_original:.4f}**")
best_r2_previous_models = max(best_ridge_r2_original, best_lasso_r2_original, best_r2_stepwise_original)


print("\n--- Comparação Final de R2 ---")
print(f"Melhor R2 dos Modelos Anteriores (Original): **{best_r2_previous_models:.4f}**")
print(f"Melhor R2 do Modelo Lasso com Features Expandidas: **{best_enhanced_lasso_r2:.4f}**")

if best_enhanced_lasso_r2 > best_r2_previous_models:
    print(f"\n**Sucesso! O R2 na base de testes foi melhorado em {best_enhanced_lasso_r2 - best_r2_previous_models:.4f}.**")
    print("As novas features e transformações contribuíram para um modelo mais preditivo.")
else:
    print(f"\n**Não houve melhora significativa no R2 na base de testes.**")
    print("O R2 do modelo com features expandidas é {best_enhanced_lasso_r2:.4f}, enquanto o melhor anterior foi {best_r2_previous_models:.4f}.")
    print("Isso pode indicar que as transformações adicionadas não capturaram novas relações importantes ou introduziram complexidade desnecessária.")


--- Previsão de Renda II: Tentando Melhorar o R2 com Engenharia de Features ---
Base de dados preparada. 12427 linhas para modelagem.
Dados divididos: Treinamento (9320 amostras), Teste (3107 amostras)

Nova fórmula com features expandidas:
log_renda ~ sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + tipo_renda + educacao + estado_civil + tipo_residencia + idade + tempo_emprego + qt_pessoas_residencia + I(idade**2) + I(tempo_emprego**2) + I(qtd_filhos**2) + idade:tempo_emprego + idade:qtd_filhos + posse_de_veiculo:idade + sexo:tempo_emprego + tem_filhos_e_casado + possui_bens

Matrizes de design ENHANCED criadas com Patsy:
X_train_enhanced_patsy shape: (9320, 33)
X_test_enhanced_patsy shape: (3107, 33)

--- Treinando e Avaliando o Modelo Lasso com Novas Features ---
  Lasso Enhanced (alpha=0.001): R2 na base de testes = 0.3868
  Lasso Enhanced (alpha=0.005): R2 na base de testes = 0.3877
  Lasso Enhanced (alpha=0.01): R2 na base de testes = 0.3860
  Lasso Enhanced (alpha=0.05):

In [59]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeRegressor # Importar a Árvore de Regressão
import warnings

warnings.filterwarnings('ignore')

print("--- Previsão de Renda II: Ajustando Árvore de Regressão ---")

# Re-carregar e preparar a base de dados
try:
    df = pd.read_csv('previsao_de_renda.csv')
    df = df.drop(columns=['Unnamed: 0', 'id_cliente', 'data_ref'], errors='ignore')
    df = df[df['renda'] > 0].copy()
    df_clean = df.dropna(subset=['tempo_emprego']).copy()
    df_clean = df_clean[df_clean['tempo_emprego'] > 0].copy()
    df_clean['log_renda'] = np.log(df_clean['renda'])
    df_clean.dropna(inplace=True)
    print(f"Base de dados preparada. {len(df_clean)} linhas para modelagem.")

except Exception as e:
    print(f"Erro na preparação da base de dados: {e}")
    exit()

# Converter variáveis categóricas em dummies (manual para DecisionTree)
# DecisionTreeRegressor não usa Patsy diretamente como Statsmodels
df_processed = pd.get_dummies(df_clean.drop(columns=['renda']), drop_first=True)

# Separar X e y
X = df_processed.drop(columns=['log_renda'])
y = df_processed['log_renda']

# Dividir em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Dados divididos: Treinamento ({len(X_train)} amostras), Teste ({len(X_test)} amostras)")
print(f"Número de features após dummyficação: {X_train.shape[1]}")

# --- Otimização de Hiperparâmetros com GridSearchCV ---
print("\n--- Otimizando Hiperparâmetros da Árvore de Regressão com GridSearchCV ---")

# Parâmetros para buscar
param_grid = {
    'max_depth': [None, 5, 10, 15, 20], # Nível máximo da árvore
    'min_samples_leaf': [1, 5, 10, 20, 50] # Número mínimo de amostras para uma folha
}

# Inicializar o modelo de Árvore de Regressão
dt_regressor = DecisionTreeRegressor(random_state=42)

# Configurar o GridSearchCV
grid_search = GridSearchCV(estimator=dt_regressor, param_grid=param_grid, 
                           cv=5, scoring='r2', n_jobs=-1, verbose=1)

# Executar a busca
grid_search.fit(X_train, y_train)

print("\n--- Resultados do GridSearchCV ---")
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Melhor R2 médio da validação cruzada: {grid_search.best_score_:.4f}")

# Obter o melhor modelo
best_dt_model = grid_search.best_estimator_

# Avaliar o melhor modelo na base de testes
y_pred_dt_test = best_dt_model.predict(X_test)
r2_dt_test = r2_score(y_test, y_pred_dt_test)

print(f"\n**R2 na Base de Testes com a Melhor Árvore de Regressão: {r2_dt_test:.4f}**")

# --- Comparação com o Melhor R2 anterior ---
# Para garantir a comparação precisa, vou recalcular o melhor R2 dos modelos anteriores
# (Ridge, Lasso, Stepwise originais) conforme o script anterior.

# Recalculando o melhor R2 dos modelos anteriores
# Definir a fórmula inicial do modelo completo (usada nos modelos anteriores)
initial_formula_rhs_original = (
    'sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + '
    'tipo_renda + educacao + estado_civil + tipo_residencia + idade + '
    'tempo_emprego + qt_pessoas_residencia'
)

# Pré-processamento Patsy para as features originais (apenas para obter os R2s anteriores)
from patsy import dmatrix, build_design_matrices
X_train_original_patsy_temp = dmatrix(initial_formula_rhs_original, data=df_clean.loc[X_train.index], return_type='dataframe')
design_info_original = X_train_original_patsy_temp.design_info

X_train_original_patsy = build_design_matrices([design_info_original], df_clean.loc[X_train.index])[0]
X_test_original_patsy = build_design_matrices([design_info_original], df_clean.loc[X_test.index])[0]

X_train_original_patsy = pd.DataFrame(X_train_original_patsy, columns=X_train_original_patsy.design_info.column_names)
X_test_original_patsy = pd.DataFrame(X_test_original_patsy, columns=X_test_original_patsy.design_info.column_names)

if 'Intercept' in X_train_original_patsy.columns:
    X_train_original_patsy = X_train_original_patsy.drop(columns='Intercept')
    X_test_original_patsy = X_test_original_patsy.drop(columns='Intercept')

# Melhores R2 dos modelos anteriores (re-calculando para garantir)
alphas_original_models = [0, 0.001, 0.005, 0.01, 0.05, 0.1]

# Melhor Ridge (original)
from sklearn.linear_model import Ridge, Lasso # Importar novamente se não estiver no escopo
ridge_results_original = []
for alpha_ridge_original in alphas_original_models:
    model_ridge_original = Ridge(alpha=alpha_ridge_original, random_state=42)
    model_ridge_original.fit(X_train_original_patsy, y_train.values.ravel())
    y_pred_test_ridge_original = model_ridge_original.predict(X_test_original_patsy)
    r2_test_ridge_original = r2_score(y_test.values.ravel(), y_pred_test_ridge_original)
    ridge_results_original.append({'alpha': alpha_ridge_original, 'r2_test': r2_test_ridge_original})
best_ridge_r2_original = pd.DataFrame(ridge_results_original)['r2_test'].max()

# Melhor Lasso (original)
lasso_results_original = []
for alpha_lasso_original in alphas_original_models:
    model_lasso_original = Lasso(alpha=alpha_lasso_original, random_state=42, max_iter=5000)
    model_lasso_original.fit(X_train_original_patsy, y_train.values.ravel())
    y_pred_test_lasso_original = model_lasso_original.predict(X_test_original_patsy)
    r2_test_lasso_original = r2_score(y_test.values.ravel(), y_pred_test_lasso_original)
    lasso_results_original.append({'alpha': alpha_lasso_original, 'r2_test': r2_test_lasso_original})
best_lasso_r2_original = pd.DataFrame(lasso_results_original)['r2_test'].max()

# Melhor Stepwise (original)
# Replicando a lógica do Stepwise para obter o R2 exato da última execução
current_vars_in_formula_original_stepwise = [term.strip() for term in initial_formula_rhs_original.split('+')]
best_r2_stepwise_original = -np.inf
import statsmodels.api as sm # Importar novamente para statsmodels

X_train_temp_stepwise = df_clean.loc[X_train.index].copy()
X_test_temp_stepwise = df_clean.loc[X_test.index].copy()

while True:
    # Patsy formula creation
    current_formula = 'log_renda ~ ' + ' + '.join(current_vars_in_formula_original_stepwise)
    if not current_vars_in_formula_original_stepwise:
        current_formula = 'log_renda ~ 1' # Just intercept if no variables left

    try:
        model_sm_original = smf.ols(formula=current_formula, data=pd.concat([X_train_temp_stepwise, y_train], axis=1)).fit()
    except Exception as e:
        print(f"Erro ao ajustar modelo OLS Stepwise: {e}")
        best_r2_stepwise_original = 0.0 # Define um valor baixo em caso de erro para não impactar a comparação
        break # Sai do loop se houver erro no ajuste do modelo

    # Predict on test data
    y_pred_test_sm_original = model_sm_original.predict(X_test_temp_stepwise)
    r2_test_current_sm_original = r2_score(y_test, y_pred_test_sm_original)

    p_values_original = model_sm_original.pvalues.drop('Intercept', errors='ignore') # Usar 'Intercept' para statsmodels.formula.api
    p_values_original = p_values_original.replace([np.inf, -np.inf], np.nan).dropna()
    non_significant_vars_original = p_values_original[p_values_original >= 0.05]

    if non_significant_vars_original.empty or not current_vars_in_formula_original_stepwise:
        best_r2_stepwise_original = r2_test_current_sm_original
        break
    else:
        var_to_remove_patsy_name_original = non_significant_vars_original.idxmax()
        
        # Encontrar a variável original correspondente à feature Patsy
        original_var_name_to_remove_original = None
        for original_var_candidate in current_vars_in_formula_original_stepwise:
            # Check for exact match or startswith for dummy variables
            if var_to_remove_patsy_name_original == original_var_candidate or \
               var_to_remove_patsy_name_original.startswith(original_var_candidate + '['):
                original_var_name_to_remove_original = original_var_candidate
                break
        
        if original_var_name_to_remove_original is None:
            # Fallback for complex Patsy transforms if not matched above
            # This is a heuristic, may need refinement based on specific Patsy output
            original_var_name_to_remove_original = var_to_remove_patsy_name_original.split('[')[0].split(':')[0]
            if original_var_name_to_remove_original not in current_vars_in_formula_original_stepwise:
                 # If still not found in current_vars, something is off, break or handle
                 best_r2_stepwise_original = r2_test_current_sm_original
                 break


        if original_var_name_to_remove_original in current_vars_in_formula_original_stepwise:
            current_vars_in_formula_original_stepwise.remove(original_var_name_to_remove_original)
        else:
            # This case means the variable was already removed or logic failed, break
            best_r2_stepwise_original = r2_test_current_sm_original
            break
            

best_r2_previous_models = max(best_ridge_r2_original, best_lasso_r2_original, best_r2_stepwise_original)

print(f"\n--- Comparação Final de R2 ---")
print(f"Melhor R2 dos Modelos Lineares (Ridge, Lasso, Stepwise Originais): **{best_r2_previous_models:.4f}**")
print(f"Melhor R2 da Árvore de Regressão na Base de Testes: **{r2_dt_test:.4f}**")

if r2_dt_test > best_r2_previous_models:
    print(f"\n**Sucesso! A Árvore de Regressão obteve um R2 melhor na base de testes.**")
    print(f"Melhoria de {r2_dt_test - best_r2_previous_models:.4f} pontos.")
    print("Isso sugere que o modelo de árvore conseguiu capturar melhor as relações não lineares e interações nos dados.")
else:
    print(f"\n**Não houve melhora significativa no R2 da base de testes com a Árvore de Regressão.**")
    print("Isso pode indicar que os modelos lineares já estavam performando bem ou que as relações no dataset são predominantemente lineares.")
    print("A diferença é de {r2_dt_test - best_r2_previous_models:.4f} pontos.")

--- Previsão de Renda II: Ajustando Árvore de Regressão ---
Base de dados preparada. 12427 linhas para modelagem.
Dados divididos: Treinamento (9320 amostras), Teste (3107 amostras)
Número de features após dummyficação: 24

--- Otimizando Hiperparâmetros da Árvore de Regressão com GridSearchCV ---
Fitting 5 folds for each of 25 candidates, totalling 125 fits

--- Resultados do GridSearchCV ---
Melhores parâmetros encontrados: {'max_depth': None, 'min_samples_leaf': 50}
Melhor R2 médio da validação cruzada: 0.3582

**R2 na Base de Testes com a Melhor Árvore de Regressão: 0.3649**
Erro ao ajustar modelo OLS Stepwise: endog has evaluated to an array with multiple columns that has shape (9320, 2). This occurs when the variable converted to endog is non-numeric (e.g., bool or str).

--- Comparação Final de R2 ---
Melhor R2 dos Modelos Lineares (Ridge, Lasso, Stepwise Originais): **0.3656**
Melhor R2 da Árvore de Regressão na Base de Testes: **0.3649**

**Não houve melhora significativa no R